## House Price Prediction with MLFlow

In [9]:
import pandas as pd
import mlflow
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score  
from sklearn.ensemble import RandomForestRegressor

AttributeError: partially initialized module 'mlflow' has no attribute 'version' (most likely due to a circular import)

In [ ]:
from sklearn.datasets import fetch_california_housing
# Load the California housing dataset
data = fetch_california_housing()
print(data.DESCR)  # Print the dataset description


In [ ]:
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='MedHouseVal')

Train test split, model hyperparameter tuning, MLFlow experiments

In [ ]:
def hyperparameter_tuning(X_train,y_train,param_grid):
    rf = RandomForestRegressor()
    grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2,scoring='neg_mean_squared_error')
    grid_search.fit(X_train, y_train)
    return grid_search

In [ ]:
from urllib.parse import urlparse
# Independent and dependent features
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
from mlflow.models import infer_signature
inferred_signature = infer_signature(X_train, y_train)

hyperparameters = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

#start mlflow experiment
mlflow.set_experiment("California_Housing_Experiment")

with mlflow.start_run(run_name="RandomForestRegressor_Hyperparameter_Tuning") as run:
    grid_search = hyperparameter_tuning(X_train, y_train, hyperparameters)
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mlflow.log_params("best_n_estimators", grid_search.best_params_['n_estimators'])
    mlflow.log_params("best_max_depth", grid_search.best_params_['max_depth'])
    mlflow.log_params("best_min_samples_split", grid_search.best_params_['min_samples_split'])
    mlflow.log_params("best_min_samples_leaf", grid_search.best_params_['min_samples_leaf'])   
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)
    mlflow.sklearn.log_model(best_model, "model", signature=inferred_signature)
    mlflow.set_tracking_uri("http://localhost:5000")
    tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
    if tracking_url_type_store != "file":
        mlflow.sklearn.log_model(best_model, "model", signature=inferred_signature, registered_model_name="Best_RandomForestRegressor_California_Housing")
    else:
        mlflow.sklearn.log_model(best_model, "model", signature=inferred_signature)
    print(f"Best Hyperparameters: {grid_search.best_params_}")
    print(f"Mean Squared Error: {mse}")
    print(f"R^2 Score: {r2}")
    print(f"Model logged in run: {run.info.run_id}")

NameError: name 'train_test_split' is not defined